In [1]:
# Part 1: Setup and Dependencies
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import nibabel as nib
import pydicom
from collections import defaultdict
import warnings
from segmentation_models_pytorch import Unet
#from data_process import MedicalSegmentationDataset
warnings.filterwarnings('ignore')

/home/s222440401/project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Data paths
DATA_PATH = '../../../../vast/s222440401'
TRAINING_PATH = DATA_PATH + '/training_images'
SEGMENTATION_PATH = DATA_PATH + '/segmentations'

print(f"Training images path: {TRAINING_PATH}")
print(f"Segmentation path: {SEGMENTATION_PATH}")


model = Unet(
    encoder_name="resnet34",        # use a ResNet encoder
    encoder_weights="imagenet",     # pretrained weights
    in_channels=1,                  # grayscale input
    classes=15                      # number of segmentation classes
)
model = model.to(device)  # Move model to GPU/CPU

# Part 4: Prepare Training Data
# Find overlapping studies
training_studies = [d for d in os.listdir(TRAINING_PATH) if os.path.isdir(os.path.join(TRAINING_PATH, d))]
segmentation_files = [f for f in os.listdir(SEGMENTATION_PATH) if f.endswith(('.nii', '.nii.gz'))]
segmentation_studies = [f.replace('.nii.gz', '').replace('.nii', '') for f in segmentation_files]
overlapping_studies = sorted(list(set(training_studies).intersection(set(segmentation_studies))))


Using device: cpu
Training images path: ../../../../vast/s222440401/training_images
Segmentation path: ../../../../vast/s222440401/segmentations


In [3]:
training_studies

['1.2.826.0.1.3680043.10001',
 '1.2.826.0.1.3680043.10005',
 '1.2.826.0.1.3680043.10014',
 '1.2.826.0.1.3680043.10016',
 '1.2.826.0.1.3680043.10032',
 '1.2.826.0.1.3680043.10041',
 '1.2.826.0.1.3680043.10051',
 '1.2.826.0.1.3680043.10058',
 '1.2.826.0.1.3680043.10062',
 '1.2.826.0.1.3680043.1010',
 '1.2.826.0.1.3680043.10136',
 '1.2.826.0.1.3680043.1016',
 '1.2.826.0.1.3680043.10179',
 '1.2.826.0.1.3680043.10204',
 '1.2.826.0.1.3680043.10230',
 '1.2.826.0.1.3680043.10261',
 '1.2.826.0.1.3680043.10360',
 '1.2.826.0.1.3680043.10400',
 '1.2.826.0.1.3680043.10412',
 '1.2.826.0.1.3680043.10431',
 '1.2.826.0.1.3680043.10443',
 '1.2.826.0.1.3680043.10449',
 '1.2.826.0.1.3680043.10515',
 '1.2.826.0.1.3680043.10520',
 '1.2.826.0.1.3680043.10541',
 '1.2.826.0.1.3680043.10579',
 '1.2.826.0.1.3680043.10606',
 '1.2.826.0.1.3680043.10608',
 '1.2.826.0.1.3680043.10614',
 '1.2.826.0.1.3680043.1062',
 '1.2.826.0.1.3680043.10628',
 '1.2.826.0.1.3680043.10633',
 '1.2.826.0.1.3680043.10639',
 '1.2.826.0.1

In [4]:
#show dcm files in a study
for study in training_studies[0:1]:
    study_path = os.path.join(TRAINING_PATH, study)
    dcm_files = [f for f in os.listdir(study_path) if f.endswith(('.dcm', '.DCM'))]
    print(dcm_files)
    print(f"Study {study} has {len(dcm_files)} DICOM files")
    #show the first 5 dcm files
    for i in range(5):
        dcm_file = os.path.join(study_path, dcm_files[i])

['1.dcm', '10.dcm', '100.dcm', '101.dcm', '102.dcm', '103.dcm', '104.dcm', '105.dcm', '106.dcm', '107.dcm', '108.dcm', '109.dcm', '11.dcm', '110.dcm', '111.dcm', '112.dcm', '113.dcm', '114.dcm', '115.dcm', '116.dcm', '117.dcm', '118.dcm', '119.dcm', '12.dcm', '120.dcm', '121.dcm', '122.dcm', '123.dcm', '124.dcm', '125.dcm', '126.dcm', '127.dcm', '128.dcm', '129.dcm', '13.dcm', '130.dcm', '131.dcm', '132.dcm', '133.dcm', '134.dcm', '135.dcm', '136.dcm', '137.dcm', '138.dcm', '139.dcm', '14.dcm', '140.dcm', '141.dcm', '142.dcm', '143.dcm', '144.dcm', '145.dcm', '146.dcm', '147.dcm', '148.dcm', '149.dcm', '15.dcm', '150.dcm', '151.dcm', '152.dcm', '153.dcm', '154.dcm', '155.dcm', '156.dcm', '157.dcm', '158.dcm', '159.dcm', '16.dcm', '160.dcm', '161.dcm', '162.dcm', '163.dcm', '164.dcm', '165.dcm', '166.dcm', '167.dcm', '168.dcm', '169.dcm', '17.dcm', '170.dcm', '171.dcm', '172.dcm', '173.dcm', '174.dcm', '175.dcm', '176.dcm', '177.dcm', '178.dcm', '179.dcm', '18.dcm', '180.dcm', '181.dcm'

In [5]:
training_studies[0:5]

['1.2.826.0.1.3680043.10001',
 '1.2.826.0.1.3680043.10005',
 '1.2.826.0.1.3680043.10014',
 '1.2.826.0.1.3680043.10016',
 '1.2.826.0.1.3680043.10032']

In [6]:
study='1.2.826.0.1.3680043.5002'
study_path = os.path.join(TRAINING_PATH, study)

In [7]:
study_path

'../../../../vast/s222440401/training_images/1.2.826.0.1.3680043.5002'

In [8]:
dcm_files = [f for f in os.listdir(study_path) if f.endswith(('.dcm', '.DCM'))]

In [9]:
print(dcm_files)

[]
